# Phase III — Empirical Implementation
## III.3 — Data Engineering

This notebook executes and validates the data-engineering layer of the extended
investment portfolio research project.

Research assumptions are governed by `docs/research_protocol.md`. This notebook
does not redefine methodological assumptions.

Current scope:

- **III.3.1 — Acquire Raw Data: IN PROGRESS**
  - Yahoo ETF acquisition/persistence: **COMPLETED**
  - FRED acquisition: **NEXT**
- **III.3.2 — Build Raw-Data Validation: IN PROGRESS**
  - Yahoo raw-data validation: **PASS**
  - FRED raw-data validation: **PENDING**
- **III.3.3 — Build the Analytical Dataset: NOT STARTED**

No monthly transformation, return calculation, portfolio construction,
macro-regime classification, rebalancing, turnover, or transaction-cost
calculation is performed in the completed Yahoo work below.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


Project root: /Users/santiago/proyectos/multifactor-portfolio-research-v2


In [2]:
import pandas as pd

from src.config import (
    ANALYSIS_CONFIG,
    RAW_MARKET_DATA_FILE,
)

from src.data.download_data import (
    download_market_data,
    get_market_data_start_date,
    get_market_data_end_date,
    save_raw_market_data,
)

from src.data.load_data import load_raw_market_data

from src.portfolio.portfolio_definitions import ELIGIBLE_ETFS

print("Data-engineering imports PASS")


Data-engineering imports PASS


## Data-Engineering Contract

The formal data-engineering specification and ownership contract was established
upstream and governs the implementation below.

The contract defines:

- canonical external data sources and identifiers;
- raw versus processed data boundaries;
- module ownership;
- analytical coverage rules;
- monthly observation policy;
- missing-data policy;
- validation responsibilities;
- blocker conditions;
- downstream exclusions.

Raw data preserves source evidence. Data-quality anomalies are detected and
investigated rather than silently repaired.


## III.3.1 — Acquire Raw Data

**Status: IN PROGRESS**

The formal acquisition stage requires both market and macroeconomic raw inputs.

Current state:

- Yahoo Finance ETF acquisition/persistence: **COMPLETED**
- FRED macroeconomic acquisition: **NEXT**

### Yahoo Finance Market Data

Yahoo Finance supplies daily market observations for the canonical ETF universe:

- SPY
- MTUM
- USMV
- QUAL
- AGG

Acquisition uses an input buffer beginning one calendar month before the
analytical sample so that the first potentially valid July 2013 monthly return
can later be constructed.

Yahoo's exclusive `end` convention is handled by the acquisition layer.
Source-level missing observations are preserved and automatic Yahoo repair is disabled.


In [3]:
print("Analysis start:", ANALYSIS_CONFIG.start_date)
print("Analysis end:", ANALYSIS_CONFIG.end_date)

print("Acquisition start:", get_market_data_start_date())
print("Exclusive acquisition end:", get_market_data_end_date())

print("Canonical ETF universe:", ELIGIBLE_ETFS)


Analysis start: 2013-07-01
Analysis end: 2024-12-31
Acquisition start: 2013-06-01
Exclusive acquisition end: 2025-01-01
Canonical ETF universe: ('SPY', 'MTUM', 'USMV', 'QUAL', 'AGG')


### Controlled Yahoo Acquisition


In [4]:
market_data = download_market_data()

print("Shape:", market_data.shape)
print("Start observation:", market_data.index.min())
print("End observation:", market_data.index.max())

print(
    "Returned tickers:",
    list(dict.fromkeys(market_data.columns.get_level_values(0)))
)

print(
    "Returned fields:",
    list(dict.fromkeys(market_data.columns.get_level_values(1)))
)


Shape: (2916, 40)
Start observation: 2013-06-03 00:00:00
End observation: 2024-12-31 00:00:00
Returned tickers: ['USMV', 'QUAL', 'MTUM', 'AGG', 'SPY']
Returned fields: ['Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits', 'Capital Gains']


### Raw Market-Data Persistence

The acquired Yahoo dataset is persisted without analytical transformation.

The raw artifact must preserve:

- dates;
- ticker identity;
- field structure;
- source-level missing values;
- corporate-action fields;
- numerical observations.

The configured artifact is stored as Parquet under `RAW_DATA_DIR`.


In [5]:
save_raw_market_data(market_data)

print("Raw file:", RAW_MARKET_DATA_FILE)
print("Yahoo raw-data persistence completed")


Raw file: /Users/santiago/proyectos/multifactor-portfolio-research-v2/data/raw/yahoo_market_data.parquet
Yahoo raw-data persistence completed


## III.3.2 — Build Raw-Data Validation

**Status: IN PROGRESS**

Yahoo raw-data validation is completed below. FRED raw-data validation remains
pending, so formal III.3.2 does not close yet.

Validation checks source structure, coverage, missingness, and persistence
preservation without silently repairing the source data.


### Yahoo Schema and Universe Validation


In [6]:
returned_tickers = set(
    market_data.columns.get_level_values(0)
)

expected_tickers = set(ELIGIBLE_ETFS)

assert returned_tickers == expected_tickers
assert isinstance(market_data.columns, pd.MultiIndex)
assert market_data.columns.nlevels == 2

print("Canonical ETF membership PASS")
print("MultiIndex schema PASS")
print("Observed width:", market_data.shape[1])


Canonical ETF membership PASS
MultiIndex schema PASS
Observed width: 40


### Early-History and Missingness Inspection

Missing or unavailable source observations are treated as data-quality events,
not automatically repaired.

QUAL begins later than the other ETFs in the acquisition window. Its unavailable
pre-inception observations are intentionally preserved.

No forward filling, interpolation, synthetic reconstruction, zero-return
replacement, or weight renormalization is performed here.


In [7]:
close_missing_counts = {
    ticker: int(market_data[ticker]["Close"].isna().sum())
    for ticker in ELIGIBLE_ETFS
}

first_valid_close = {
    ticker: market_data[ticker]["Close"].first_valid_index()
    for ticker in ELIGIBLE_ETFS
}

print("Missing Close observations:")
for ticker, count in close_missing_counts.items():
    print(f"{ticker}: {count}")

print("\nFirst valid Close observation:")
for ticker, first_date in first_valid_close.items():
    print(f"{ticker}: {first_date}")


Missing Close observations:
SPY: 0
MTUM: 0
USMV: 0
QUAL: 32
AGG: 0

First valid Close observation:
SPY: 2013-06-03 00:00:00
MTUM: 2013-06-03 00:00:00
USMV: 2013-06-03 00:00:00
QUAL: 2013-07-18 00:00:00
AGG: 2013-06-03 00:00:00


### Raw Persistence Reload


In [8]:
reloaded_market_data = load_raw_market_data()

print("Original shape:", market_data.shape)
print("Reloaded shape:", reloaded_market_data.shape)

assert market_data.shape == reloaded_market_data.shape

print("Reload shape preservation PASS")


Original shape: (2916, 40)
Reloaded shape: (2916, 40)
Reload shape preservation PASS


### Parquet Timestamp-Resolution Observation

During validation, the Parquet round trip changed the pandas `DatetimeIndex`
resolution representation from `datetime64[s]` to `datetime64[ms]`.

Investigation confirmed that the timestamp values themselves were unchanged.

This is classified as a storage-representation difference rather than empirical
data loss.

For preservation validation only, both indices are converted to a common
`datetime64[ns]` representation before strict DataFrame equality is tested.
No source observation is altered in the persisted raw dataset.


In [9]:
print("Original index dtype:", market_data.index.dtype)
print("Reloaded index dtype:", reloaded_market_data.index.dtype)

print("Original first date:", market_data.index[0])
print("Reloaded first date:", reloaded_market_data.index[0])

print("Original last date:", market_data.index[-1])
print("Reloaded last date:", reloaded_market_data.index[-1])

print(
    "Same index length:",
    len(market_data.index) == len(reloaded_market_data.index),
)


Original index dtype: datetime64[s]
Reloaded index dtype: datetime64[ms]
Original first date: 2013-06-03 00:00:00
Reloaded first date: 2013-06-03 00:00:00
Original last date: 2024-12-31 00:00:00
Reloaded last date: 2024-12-31 00:00:00
Same index length: True


### Timestamp-Value and Full-Frame Preservation


In [10]:
original_for_validation = market_data.copy()
reloaded_for_validation = reloaded_market_data.copy()

original_for_validation.index = (
    original_for_validation.index.astype("datetime64[ns]")
)

reloaded_for_validation.index = (
    reloaded_for_validation.index.astype("datetime64[ns]")
)

pd.testing.assert_index_equal(
    original_for_validation.index,
    reloaded_for_validation.index,
)

pd.testing.assert_frame_equal(
    original_for_validation,
    reloaded_for_validation,
)

print("Index timestamp values PASS")
print("Raw market-data persistence PASS")


Index timestamp values PASS
Raw market-data persistence PASS


### Targeted Preservation Checks

In addition to full-frame equality, two research-specific properties are checked
explicitly:

1. QUAL's early missing observations survive persistence unchanged.
2. Yahoo corporate-action fields remain present after reload.


In [11]:
original_qual_na = market_data["QUAL"]["Close"].isna().sum()
reloaded_qual_na = reloaded_market_data["QUAL"]["Close"].isna().sum()

assert original_qual_na == reloaded_qual_na

print(
    "QUAL missingness preservation PASS:",
    original_qual_na,
)


QUAL missingness preservation PASS: 32


In [12]:
required_action_fields = {
    "Dividends",
    "Stock Splits",
    "Capital Gains",
}

original_fields = set(
    market_data.columns.get_level_values(1)
)

reloaded_fields = set(
    reloaded_market_data.columns.get_level_values(1)
)

assert required_action_fields.issubset(original_fields)
assert required_action_fields.issubset(reloaded_fields)

print("Corporate-action fields preservation PASS")


Corporate-action fields preservation PASS


### Yahoo Raw-Data Checkpoint


In [13]:
print("Yahoo Raw Acquisition & Validation")
print("----------------------------------")
print("Yahoo acquisition                 PASS")
print("Canonical ETF universe            PASS")
print("Raw persistence                   PASS")
print("Timestamp values preserved        PASS")
print("QUAL missingness preserved        PASS")
print("Corporate-action fields           PASS")
print("Prohibited imputation             NONE")
print("Automatic Yahoo repair            DISABLED")
print()
print("Yahoo raw-data checkpoint: COMPLETED")


Yahoo Raw Acquisition & Validation
----------------------------------
Yahoo acquisition                 PASS
Canonical ETF universe            PASS
Raw persistence                   PASS
Timestamp values preserved        PASS
QUAL missingness preserved        PASS
Corporate-action fields           PASS
Prohibited imputation             NONE
Automatic Yahoo repair            DISABLED

Yahoo raw-data checkpoint: COMPLETED


## Yahoo Raw-Data Checkpoint — Result

**COMPLETED**

Yahoo Finance acquisition, persistence, reload, and raw-data validation have
passed.

The validated Yahoo raw-data work preserves:

- daily observation dates;
- canonical ticker identities;
- MultiIndex field structure;
- numerical observations;
- source-level missing values;
- QUAL's early unavailable observations;
- corporate-action fields.

A Parquet timestamp-resolution difference (`datetime64[s]` → `datetime64[ms]`)
was observed and investigated. Timestamp values were confirmed identical after
normalizing resolution solely for validation.

No cleaning, imputation, monthly transformation, return calculation, portfolio
logic, or analytical calculation occurred during persistence or raw validation.

### Remaining work before III.3.1 / III.3.2 can close

- Acquire `DGS3MO`.
- Acquire `T10Y3M`.
- Persist raw FRED observations.
- Validate FRED schema, coverage, missingness, ordering, and persistence preservation.

Only after both raw market and macro inputs have passed acquisition and raw-data
validation should the project proceed to:

> **III.3.3 — Build the Analytical Dataset**

Macro-regime classification and the one-month regime lag remain downstream and
must not be implemented during raw FRED acquisition.
